In [1]:
from pathlib import Path
import json
import urllib.request

import pandas as pd

SOURCE_URL = (
    "https://github.com/DataTalksClub/nyc-tlc-data/releases/download/"
    "yellow/yellow_tripdata_2021-01.csv.gz"
)
DATA_PATH = Path("yellow_tripdata_2021-01.csv.gz")
CHUNK_SIZE = 100_000
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Pandas version:", pd.__version__)
print("Working directory:", Path.cwd())

Pandas version: 2.3.3
Working directory: c:\Users\DELL\Desktop\L4-Semster 1\Data Engineering\Lab3_NYC_Taxi_Project.ipynb\Lab3_NYC_Taxi


In [2]:
if not DATA_PATH.exists():
    print("Downloading dataset...")
    urllib.request.urlretrieve(SOURCE_URL, DATA_PATH)

print("Dataset path:", DATA_PATH.resolve())
print("File exists:", DATA_PATH.exists())

Dataset path: C:\Users\DELL\Desktop\L4-Semster 1\Data Engineering\Lab3_NYC_Taxi_Project.ipynb\Lab3_NYC_Taxi\yellow_tripdata_2021-01.csv.gz
File exists: True


In [3]:
preview_df = pd.read_csv(DATA_PATH, compression="gzip", nrows=100)

print("=== df.head() ===")
print(preview_df.head())

print("\n=== df.dtypes ===")
print(preview_df.dtypes)

print("\nPreview shape:", preview_df.shape)

=== df.head() ===
   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         1  2021-01-01 00:30:10   2021-01-01 00:36:12                1   
1         1  2021-01-01 00:51:20   2021-01-01 00:52:19                1   
2         1  2021-01-01 00:43:30   2021-01-01 01:11:06                1   
3         1  2021-01-01 00:15:48   2021-01-01 00:31:01                0   
4         2  2021-01-01 00:31:49   2021-01-01 00:48:21                1   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0           2.10           1                  N           142            43   
1           0.20           1                  N           238           151   
2          14.70           1                  N           132           165   
3          10.60           1                  N           138           132   
4           4.94           1                  N            68            33   

   payment_type  fare_amount  extra  mta_tax  tip_amount

In [4]:
def is_january_2021(series):
    """True when a pickup timestamp parses and falls in January 2021."""
    dates = pd.to_datetime(series, errors="coerce")
    return dates.notna() & dates.ge("2021-01-01") & dates.lt("2021-02-01")

In [5]:
dtype_map = {}
null_counts = {}
minima = {}
maxima = {}

rows_total = 0
bad_pickup_count = 0
fare_nonpositive_count = 0
distance_nonpositive_count = 0
passenger_zero_count = 0
passenger_null_count = 0
duplicate_row_count = 0
seen_hashes = set()

reader = pd.read_csv(
    DATA_PATH,
    compression="gzip",
    chunksize=CHUNK_SIZE
)

for chunk_number, chunk in enumerate(reader, start=1):
    rows_total += len(chunk)

    for col in chunk.columns:
        dtype_map.setdefault(col, str(chunk[col].dtype))
        null_counts[col] = null_counts.get(col, 0) + int(chunk[col].isna().sum())

        values = chunk[col].dropna()
        if not values.empty:
            try:
                local_min = values.min()
                local_max = values.max()
                if col not in minima or local_min < minima[col]:
                    minima[col] = local_min
                if col not in maxima or local_max > maxima[col]:
                    maxima[col] = local_max
            except (TypeError, ValueError):
                # Some columns may not support min/max comparisons.
                pass

    if "tpep_pickup_datetime" in chunk.columns:
        bad_pickup_count += int((~is_january_2021(chunk["tpep_pickup_datetime"])).sum())

    if "fare_amount" in chunk.columns:
        fare_nonpositive_count += int((chunk["fare_amount"] <= 0).fillna(False).sum())

    if "trip_distance" in chunk.columns:
        distance_nonpositive_count += int((chunk["trip_distance"] <= 0).fillna(False).sum())

    if "passenger_count" in chunk.columns:
        passenger_zero_count += int((chunk["passenger_count"] == 0).fillna(False).sum())
        passenger_null_count += int(chunk["passenger_count"].isna().sum())

    row_hashes = pd.util.hash_pandas_object(chunk, index=False).astype("uint64")
    for value in row_hashes:
        row_hash = int(value)
        if row_hash in seen_hashes:
            duplicate_row_count += 1
        else:
            seen_hashes.add(row_hash)

    print(
        f"Chunk {chunk_number}: {len(chunk):,} rows | "
        f"cumulative rows: {rows_total:,}"
    )

print("\nTotal rows scanned:", f"{rows_total:,}")
print("Reference count from class slides: 1,369,765")

Chunk 1: 100,000 rows | cumulative rows: 100,000
Chunk 2: 100,000 rows | cumulative rows: 200,000
Chunk 3: 100,000 rows | cumulative rows: 300,000
Chunk 4: 100,000 rows | cumulative rows: 400,000
Chunk 5: 100,000 rows | cumulative rows: 500,000
Chunk 6: 100,000 rows | cumulative rows: 600,000
Chunk 7: 100,000 rows | cumulative rows: 700,000
Chunk 8: 100,000 rows | cumulative rows: 800,000
Chunk 9: 100,000 rows | cumulative rows: 900,000
Chunk 10: 100,000 rows | cumulative rows: 1,000,000
Chunk 11: 100,000 rows | cumulative rows: 1,100,000
Chunk 12: 100,000 rows | cumulative rows: 1,200,000


C:\Users\DELL\AppData\Local\Temp\ipykernel_1676\1283755613.py:21: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk_number, chunk in enumerate(reader, start=1):


Chunk 13: 100,000 rows | cumulative rows: 1,300,000
Chunk 14: 69,765 rows | cumulative rows: 1,369,765

Total rows scanned: 1,369,765
Reference count from class slides: 1,369,765


## 6. Print the column profiling table
Every table is printed using `print()`, as requested.

In [6]:
column_profile = pd.DataFrame({
    "column": list(dtype_map.keys()),
    "dtype": [dtype_map[c] for c in dtype_map],
    "null_count": [null_counts.get(c, 0) for c in dtype_map],
})
column_profile["null_percent"] = (
    column_profile["null_count"] / rows_total * 100
).round(4)
column_profile["min"] = column_profile["column"].map(minima)
column_profile["max"] = column_profile["column"].map(maxima)

print(column_profile.to_string(index=False))
column_profile.to_csv(OUTPUT_DIR / "column_profile.csv", index=False)

               column   dtype  null_count  null_percent                 min                 max
             VendorID   int64       98352        7.1802                   1                   2
 tpep_pickup_datetime  object           0        0.0000 2008-12-31 23:05:14 2021-02-22 16:52:16
tpep_dropoff_datetime  object           0        0.0000 2008-12-31 23:07:22 2021-02-22 16:56:15
      passenger_count   int64       98352        7.1802                   0                   8
        trip_distance float64           0        0.0000                 0.0           263163.28
           RatecodeID   int64       98352        7.1802                   1                  99
   store_and_fwd_flag  object       98352        7.1802                   N                   Y
         PULocationID   int64           0        0.0000                   1                 265
         DOLocationID   int64           0        0.0000                   1                 265
         payment_type   int64       9835

In [7]:
quality_counts = pd.DataFrame([
    {"issue": "Pickup outside January 2021 or invalid", "count": bad_pickup_count},
    {"issue": "fare_amount <= 0", "count": fare_nonpositive_count},
    {"issue": "trip_distance <= 0", "count": distance_nonpositive_count},
    {"issue": "passenger_count = 0", "count": passenger_zero_count},
    {"issue": "passenger_count is null", "count": passenger_null_count},
    {"issue": "Exact duplicate rows (hash-based)", "count": duplicate_row_count},
])

print(quality_counts.to_string(index=False))
quality_counts.to_csv(OUTPUT_DIR / "data_quality_counts.csv", index=False)

summary = {
    "rows_total": int(rows_total),
    "chunk_size": CHUNK_SIZE,
    "quality_counts": {
        row["issue"]: int(row["count"])
        for _, row in quality_counts.iterrows()
    },
}
with open(OUTPUT_DIR / "profiling_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("\nSaved profiling tables to:", OUTPUT_DIR.resolve())

                                 issue  count
Pickup outside January 2021 or invalid     24
                      fare_amount <= 0   7411
                    trip_distance <= 0  19952
                   passenger_count = 0  26726
               passenger_count is null  98352
     Exact duplicate rows (hash-based)      0

Saved profiling tables to: C:\Users\DELL\Desktop\L4-Semster 1\Data Engineering\Lab3_NYC_Taxi_Project.ipynb\Lab3_NYC_Taxi\outputs


In [8]:
def clean(df):
    """Return (cleaned_dataframe, rows_removed_per_rule) without modifying df."""
    cleaned_df = df.copy()
    rows_removed = {}

    before = len(cleaned_df)
    cleaned_df = cleaned_df.loc[cleaned_df["passenger_count"].notna()].copy()
    rows_removed["null_passenger_count"] = before - len(cleaned_df)

    before = len(cleaned_df)
    valid_pickups = is_january_2021(cleaned_df["tpep_pickup_datetime"])
    cleaned_df = cleaned_df.loc[valid_pickups].copy()
    rows_removed["pickup_outside_january_2021_or_invalid"] = before - len(cleaned_df)

    before = len(cleaned_df)
    cleaned_df = cleaned_df.loc[cleaned_df["fare_amount"] > 0].copy()
    rows_removed["fare_amount_less_than_or_equal_to_zero"] = before - len(cleaned_df)

    return cleaned_df, rows_removed

## 9. Demonstrate cleaning and print before/after counts
To keep memory usage low, this demonstration applies `clean(df)` to the first 100,000-row chunk. The function itself can be used on any DataFrame/chunk.

In [9]:
first_chunk = next(pd.read_csv(
    DATA_PATH,
    compression="gzip",
    chunksize=CHUNK_SIZE
))
original_first_chunk = first_chunk.copy(deep=True)

cleaned_chunk, removed_rows = clean(first_chunk)

rows_before = len(first_chunk)
rows_after = len(cleaned_chunk)
rows_removed_total = rows_before - rows_after

print("Rows before:", f"{rows_before:,}")
print("Rows after:", f"{rows_after:,}")
print("Rows removed total:", f"{rows_removed_total:,}")
print("Rows removed per rule:")
print(removed_rows)
print("Original input unchanged:", first_chunk.equals(original_first_chunk))

cleaning_result = {
    "scope": "first chunk only",
    "rows_before": int(rows_before),
    "rows_after": int(rows_after),
    "rows_removed_total": int(rows_removed_total),
    "rows_removed_per_rule": removed_rows,
}
with open(OUTPUT_DIR / "cleaning_demo.json", "w", encoding="utf-8") as f:
    json.dump(cleaning_result, f, indent=2)

Rows before: 100,000
Rows after: 99,263
Rows removed total: 737
Rows removed per rule:
{'null_passenger_count': 0, 'pickup_outside_january_2021_or_invalid': 16, 'fare_amount_less_than_or_equal_to_zero': 721}
Original input unchanged: True
